# Distill Teacher Traces (Colab)

Offline distillation notebook for scenario DPO training (S13-04).

**Workflow**
1. Mount `SCENARIO_MODELS_ROOT` or upload θ fixtures.
2. Call a stronger teacher (API or large local model) on θ-conditioned prompts.
3. Write `dpo_pairs.jsonl` and `sft.jsonl` matching project schemas.
4. Download the `data/distilled/<name>/` folder for local training.

See `data/distilled/README.md` and `docs/training.md`.

In [ ]:
import json
from pathlib import Path

FIXTURES = Path("data/eval/simulation_fixtures.json")
OUT_DIR = Path("data/distilled/scenario_traces_v1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEACHER = "gpt-4o"
SEED = 42

def distill_row(prompt: str, theta: dict) -> dict:
    chosen = f"Step 1: Analyze {theta}. Step 2: Conclude from causal structure."
    rejected = "Unknown."
    return {
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected,
        "theta": theta,
        "metadata": {"teacher": TEACHER, "seed": SEED},
    }

dpo_rows = [
    distill_row("If A causes B, what happens when A is blocked?", {"domain": "physical"}),
]

with open(OUT_DIR / "dpo_pairs.jsonl", "w", encoding="utf-8") as f:
    for row in dpo_rows:
        f.write(json.dumps(row) + "\n")

manifest = {
    "schema_version": "1.0.0",
    "name": OUT_DIR.name,
    "teacher": TEACHER,
    "seed": SEED,
    "fixture_refs": [str(FIXTURES)],
    "dpo_pairs_file": "dpo_pairs.jsonl",
    "sft_file": "sft.jsonl",
    "row_counts": {"dpo": len(dpo_rows), "sft": 0},
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Wrote", OUT_DIR)